# Interactive Trick Play

This notebook allows you to play a single trick interactively in Jupyter.

You can:
- View your hand
- See the current trick state
- Select a card to play
- See the result of the trick


In [1]:
import sys
from pathlib import Path

# Add project root to path
project_root = Path().resolve().parent
sys.path.insert(0, str(project_root))

from eucher.cards import Card, Deck, Suit
from eucher.game import Game
from eucher.rules import RulesEngine
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets
from typing import List, Optional


In [2]:
class InteractiveTrick:
    """Interactive trick play interface."""
    
    def __init__(self):
        self.deck = Deck()
        self.deck.shuffle()
        self.hand = self.deck.deal(5)
        self.trump_suit = self.deck.draw_one().suit
        self.trick_cards: List[Card] = []
        self.trick_player_ids: List[int] = []
        self.led_suit: Optional[Suit] = None
        self.rules = RulesEngine()
        self.current_player = 0
        
    def get_valid_cards(self) -> List[Card]:
        """Get valid cards that can be played."""
        return self.rules.get_valid_plays(self.hand, self.led_suit, self.trump_suit)
    
    def play_card(self, card: Card) -> None:
        """Play a card."""
        if card not in self.hand:
            raise ValueError(f"Card {card} not in hand")
        
        valid_cards = self.get_valid_cards()
        if card not in valid_cards:
            raise ValueError(f"Card {card} is not a valid play")
        
        # Remove from hand
        self.hand.remove(card)
        self.trick_cards.append(card)
        self.trick_player_ids.append(self.current_player)
        
        # Set led suit if first card
        if len(self.trick_cards) == 1:
            self.led_suit = card.suit
        
        self.current_player += 1
    
    def get_winner(self) -> Optional[int]:
        """Get the winner of the trick if complete."""
        if len(self.trick_cards) < 4:
            return None
        
        player_ids = list(range(4))
        winner_id = self.rules.determine_trick_winner(
            self.trick_cards, player_ids, self.led_suit or self.trick_cards[0].suit, self.trump_suit
        )
        return winner_id
    
    def display_state(self) -> None:
        """Display current game state."""
        html = f"""
        <div style="border: 2px solid #333; padding: 10px; margin: 10px;">
            <h3>Interactive Trick</h3>
            <p><strong>Trump Suit:</strong> {self.trump_suit.name}</p>
            <p><strong>Led Suit:</strong> {self.led_suit.name if self.led_suit else 'None'}</p>
            <p><strong>Your Hand:</strong> {', '.join(str(c) for c in self.hand)}</p>
            <p><strong>Trick Cards:</strong> {', '.join(str(c) for c in self.trick_cards) if self.trick_cards else 'None'}</p>
            <p><strong>Cards Played:</strong> {len(self.trick_cards)}/4</p>
        </div>
        """
        display(HTML(html))

# Initialize trick
trick = InteractiveTrick()
trick.display_state()


In [3]:
def create_card_buttons():
    """Create buttons for each card in hand."""
    valid_cards = trick.get_valid_cards()
    
    buttons = []
    for card in valid_cards:
        button = widgets.Button(
            description=str(card),
            button_style='info',
            layout=widgets.Layout(width='150px', height='50px')
        )
        
        def make_play_card(card_to_play):
            def on_button_click(b):
                try:
                    trick.play_card(card_to_play)
                    clear_output(wait=True)
                    trick.display_state()
                    
                    if len(trick.trick_cards) < 4:
                        # Show next set of buttons
                        display(create_card_buttons())
                    else:
                        # Trick complete
                        winner = trick.get_winner()
                        display(HTML(f"<h3>Trick Complete! Winner: Player {winner}</h3>"))
                except Exception as e:
                    display(HTML(f"<p style='color: red;'>{e}</p>"))
                    trick.display_state()
                    display(create_card_buttons())
            
            return on_button_click
        
        button.on_click(make_play_card(card))
        buttons.append(button)
    
    return widgets.HBox(buttons)

# Display card selection buttons
if len(trick.trick_cards) < 4:
    display(create_card_buttons())
